In [47]:
# Prices per million tokens (USD). Source: claude.com/pricing
PRICING = {
    "claude-opus-4-7":   {"input": 5.00, "output": 25.00},
    "claude-opus-4-6":   {"input": 5.00, "output": 25.00},
    "claude-sonnet-4-6": {"input": 3.00, "output": 15.00},
    "claude-haiku-4-5-20251001":  {"input": 1.00, "output": 5.00},
}

def cost_of_call(response, batch=False):
    rates = PRICING[response.model]
    u = response.usage

    cost = (u.input_tokens / 1_000_000) * rates["input"]
    cost += (u.output_tokens / 1_000_000) * rates["output"]

    # Cache reads are ~10% of base input price
    cache_read = getattr(u, "cache_read_input_tokens", 0) or 0
    cost += (cache_read / 1_000_000) * rates["input"] * 0.10

    # Cache writes (5-min) are ~25% above base input
    cache_write = getattr(u, "cache_creation_input_tokens", 0) or 0
    cost += (cache_write / 1_000_000) * rates["input"] * 1.25

    if batch:
        cost *= 0.5

    out = f"""
        Input tokens: {u.input_tokens}
        Output tokens: {u.output_tokens}
        Total tokens: {u.input_tokens + u.output_tokens}
        Cost: {cost}
    """
    return out


In [ ]:
# Install dependencies
%pip install anthropic python-dotenv

In [ ]:
# Load environment variables

from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Create an API cleint

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"


In [51]:
# Make a request

message = client.messages.create(
    model=model,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "How can Claude help VFX and animation pipelines? Answer in one line."
        }
    ]
)

In [45]:
message

Message(id='msg_01DHFvBrKP6qMd6TuJuya3Vg', container=None, content=[TextBlock(citations=None, text='Claude can accelerate VFX and animation pipelines by automating script generation, asset organization, reviewing frame sequences, troubleshooting technical issues, and generating documentation—allowing artists to focus on creative work.', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=24, output_tokens=46, server_tool_use=None, service_tier='standard'))

In [52]:
message.content[0].text

'Claude can streamline VFX and animation pipelines by automating script generation, analyzing footage for technical issues, assisting with shot planning, generating documentation, and helping troubleshoot complex rendering or compositing problems through technical consultation.'

In [48]:
cost_of_call(message)

'\n        Input tokens: 24\n        Output tokens: 46\n        Total tokens: 70\n        Cost: 0.000254\n    '

In [ ]:
message = client.messages.create(
    model=model,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "How can Claude help VFX and animation pipelines? Answer in one line."
        },
        {
            "role": "assistant",
            "content": "Claude can streamline VFX and animation pipelines by automating script generation, analyzing footage for technical issues, assisting with shot planning, generating documentation, and helping troubleshoot complex rendering or compositing problems through technical consultation."
        },
        {
            "role": "user",
            "content": "Can you tell me more about automating script generation? Answered in one line."
        }
    ]
)